# Chapter 3: Invariant Geometry of Probability Distributions

Source orientation: printed pages 51-70, PDF pages 63-81. This notebook is a standalone computational retelling of the chapter's invariant-geometry thread: what changes when observations are coarsened, what stays fixed when a statistic is sufficient, why decomposable invariant divergences are f-divergences, and why the Fisher metric is the invariant metric that survives those tests.


## Chapter Question

A statistical model can be drawn with many coordinate systems and many observation alphabets. If we replace a raw outcome `x` by a reported value `y = k(x)`, then every distribution on `x` pushes forward to a distribution on `y`. A geometry for probability laws should react to this operation in a disciplined way: losing observational detail should not make two distributions look more distinguishable.

This is the chapter's organizing criterion. Coarse graining may shrink divergence. It must preserve divergence exactly when the reported value is a sufficient statistic for the family under study. The same idea, made infinitesimal, selects the Fisher information metric up to scale.


In [ ]:
from pathlib import Path
import json
import numpy as np

import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import Image, HTML, display

ARTIFACT_DIR = Path('../../artifacts/chapter-03')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
EPS = 1e-12


def display_artifact(name, width=None):
    path = ARTIFACT_DIR / name
    if path.suffix.lower() in {'.png', '.jpg', '.jpeg'}:
        display(Image(filename=str(path), width=width))
    elif path.suffix.lower() == '.html':
        display(HTML(f'<a href="{path.as_posix()}" target="_blank">Open {name}</a>'))
    else:
        display(HTML(f'<a href="{path.as_posix()}" target="_blank">{name}</a>'))
    return path


def normalize(v):
    v = np.asarray(v, dtype=float)
    return v / v.sum()


def coarse(v, blocks):
    v = np.asarray(v, dtype=float)
    return np.array([v[list(block)].sum() for block in blocks], dtype=float)


def f_divergence(p, q, kind='kl'):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    if kind == 'kl':
        return float(np.sum(np.where(p > EPS, p * np.log(np.clip(p, EPS, None) / np.clip(q, EPS, None)), 0.0)))
    if kind == 'reverse_kl':
        return float(np.sum(np.where(q > EPS, q * np.log(np.clip(q, EPS, None) / np.clip(p, EPS, None)), 0.0)))
    if kind == 'hellinger2':
        return float(np.sum(2.0 * (np.sqrt(np.clip(p, 0, None)) - np.sqrt(np.clip(q, 0, None))) ** 2))
    if kind == 'pearson_chi2':
        return float(np.sum(0.5 * (p - q) ** 2 / np.clip(p, EPS, None)))
    raise ValueError(kind)


def alpha_divergence(p, q, alpha):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    if abs(alpha - 1.0) < 1e-10:
        return f_divergence(p, q, 'reverse_kl')
    if abs(alpha + 1.0) < 1e-10:
        return f_divergence(p, q, 'kl')
    return float(4.0 / (1.0 - alpha ** 2) * (1.0 - np.sum(p ** ((1 - alpha) / 2) * q ** ((1 + alpha) / 2))))


## Coarse Graining as a Markov Map

For a finite alphabet, a coarse graining is just a partition of fine outcomes. The probability of a coarse cell is the sum of the probabilities in that cell. In matrix language this is a row-stochastic map with one `1` per fine outcome. The operation is simple, but the invariant-geometry demand is strong:

$$D(Pp:Pq) \leq D(p:q).$$

Here `P` is the coarse-graining map. The inequality says that a lossy observation cannot increase distinguishability.


In [ ]:
display_artifact('coarse_graining_partition.png', width=760)


## f-Divergences: The Decomposable Invariant Family

A decomposable divergence adds one local contribution per outcome. The invariant family has the form

$$D_f(p:q)=\sum_i p_i f\!\left({q_i\over p_i}\right),$$

where `f` is convex and normalized so that `f(1)=0`. Adding a linear term to `f` does not change the divergence on the probability simplex, and the usual scale convention is `f''(1)=1`.

The proof of monotonicity is Jensen's inequality in disguise. When two fine cells are merged, the new ratio `q/p` is a weighted average of the old ratios. Convexity says the divergence after averaging cannot exceed the weighted divergence before averaging.


In [ ]:
p = normalize([0.09, 0.11, 0.18, 0.07, 0.14, 0.21, 0.08, 0.12])
q = normalize([0.05, 0.15, 0.12, 0.13, 0.10, 0.25, 0.14, 0.06])
blocks = [[0, 1], [2, 3, 4], [5, 6, 7]]
pb, qb = coarse(p, blocks), coarse(q, blocks)

kinds = ['kl', 'reverse_kl', 'hellinger2', 'pearson_chi2']
labels = ['KL', 'dual KL', 'Hellinger^2', 'Pearson chi^2']
full = np.array([f_divergence(p, q, k) for k in kinds])
coarse_vals = np.array([f_divergence(pb, qb, k) for k in kinds])

fig, ax = plt.subplots(figsize=(8.8, 4.6))
idx = np.arange(len(kinds))
width = 0.36
ax.bar(idx - width/2, full, width, label='fine distribution', color='#4C78A8')
ax.bar(idx + width/2, coarse_vals, width, label='after coarse graining', color='#E45756')
for i, gap in enumerate(full - coarse_vals):
    ax.text(i, max(full[i], coarse_vals[i]) + 0.01, f'loss {gap:.3f}', ha='center', fontsize=9)
ax.set_xticks(idx, labels)
ax.set_ylabel('divergence value')
ax.set_title('Information monotonicity for standard f-divergences')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / 'f_divergence_monotonicity.png', dpi=180, bbox_inches='tight')
plt.close(fig)
display_artifact('f_divergence_monotonicity.png', width=760)


## Equality and Sufficient Statistics

The equality case is the important diagnostic. Suppose each coarse cell has an internal conditional distribution that does not depend on the model parameter. Then observing the coarse cell loses no information about the parameter. In the finite setting this means that, inside every block, `p(x | y)` and `q(x | y)` match for the two parameter values being compared.

The next artifact contrasts that sufficient case with a nearby case where one conditional distribution inside a block is changed. The coarse distributions are the same in both comparisons, but the fine divergence detects the hidden conditional change.


In [ ]:
display_artifact('sufficient_statistic_equality.png', width=720)


## Standard Examples

The chapter's examples fit into one table:

| generator `f(u)` | divergence | geometric role |
| --- | --- | --- |
| `-log u` | KL `D(p:q)` | the `alpha=-1` endpoint |
| `u log u` | dual KL `D(q:p)` | the `alpha=1` endpoint |
| `(u-1)^2 / 2` | Pearson chi-square | a quadratic local contrast |
| `4(1 - sqrt(u))` | squared Hellinger form | the symmetric `alpha=0` case |

All standard f-divergences induce the same second-order metric. Their higher-order behavior differs, and those differences become the `alpha`-geometry developed later.


## Interactive Alpha-Divergence Landscape

The Plotly figure samples the probability simplex for a fixed reference distribution `q`. The three traces show how `alpha=-1`, `alpha=0`, and `alpha=1` score the same candidate distributions. Near `q` they share the Fisher quadratic approximation; farther away they encode different preferences about support mismatch and tail behavior.


In [ ]:
q_ref = normalize([0.45, 0.35, 0.20])
grid = []
for a in np.linspace(0.02, 0.96, 44):
    for b in np.linspace(0.02, 0.96 - a, 44):
        c = 1 - a - b
        if c >= 0.02:
            pp = np.array([a, b, c])
            grid.append((a, b, c, alpha_divergence(pp, q_ref, -1.0), alpha_divergence(pp, q_ref, 0.0), alpha_divergence(pp, q_ref, 1.0)))
arr = np.array(grid)
fig = go.Figure()
for col, name in [(3, 'alpha = -1 (KL p:q)'), (4, 'alpha = 0 (Hellinger)'), (5, 'alpha = 1 (dual KL)')]:
    fig.add_trace(go.Scatterternary(
        a=arr[:,0], b=arr[:,1], c=arr[:,2], mode='markers', name=name,
        marker={'size':5, 'color':arr[:,col], 'colorscale':'Viridis', 'showscale': col == 3, 'opacity':0.72}
    ))
fig.add_trace(go.Scatterternary(a=[q_ref[0]], b=[q_ref[1]], c=[q_ref[2]], mode='markers+text', text=['q'], textposition='top center', name='reference q', marker={'size':13, 'color':'black', 'symbol':'x'}))
fig.update_layout(title='Alpha-divergence level samples on the probability simplex', ternary={'sum':1, 'aaxis':{'title':'p0'}, 'baxis':{'title':'p1'}, 'caxis':{'title':'p2'}}, height=650)
fig.write_html(ARTIFACT_DIR / 'alpha_divergence_simplex.html', include_plotlyjs='cdn', full_html=True)
fig


## Fisher Metric Invariance

The Fisher metric on the simplex has the quadratic form

$$g_p(v,v)=\sum_i {v_i^2\over p_i}, \qquad \sum_i v_i=0.$$

A sufficient-statistic embedding splits a coarse mass `q_j` into fine masses `r_{ij} q_j`, where the conditional weights `r_{ij}` are fixed. The tangent vector splits in the same proportions. Substitution gives

$$\sum_{i\in A_j}{(r_{ij}v_j)^2\over r_{ij}q_j}= {v_j^2\over q_j}\sum_{i\in A_j}r_{ij}= {v_j^2\over q_j}.$$

So the embedded tangent has the same Fisher length. This is the computational heart of Chentsov's uniqueness statement in the finite simplex.


In [ ]:
display_artifact('fisher_embedding_invariance.png', width=720)


## Positive Measures and the Square-Root Picture

On the positive cone, the same second-order calculation gives

$$ds^2 = \sum_i {dm_i^2\over m_i}.$$

With coordinates `xi_i = 2 sqrt(m_i)`, this becomes the ordinary Euclidean metric. The probability simplex is the section `sum_i m_i = 1`, which becomes a sphere of radius `2` in square-root coordinates. This is why the Hellinger picture often feels so geometrically natural: it is the Euclidean shadow of the invariant metric on positive measures.


## Applied Lab: Sensor Bins and a Privacy-Preserving Report

Imagine a device that records eight fine states: activity level crossed with a low/high signal flag. A public report collapses those states into four bins. The report is less revealing, so every f-divergence between baseline and shifted behavior should weakly decrease. If a downstream estimator only depends on the coarse activity bin and the within-bin low/high split is parameter-free, the coarsened report is sufficient for that estimator.

The lab data are saved as `applied_lab_sensor_bins.json`. Use the cells below to compare the fine and coarse reports, then modify the block definitions to see how the reported resolution changes the information loss.


In [ ]:
lab_rows = json.loads((ARTIFACT_DIR / 'applied_lab_sensor_bins.json').read_text(encoding='utf-8'))
fine = [row for row in lab_rows if row['level'] == 'fine']
coarse_rows = [row for row in lab_rows if row['level'] == 'coarse']
lab_p = np.array([row['p_baseline'] for row in fine])
lab_q = np.array([row['q_shifted'] for row in fine])
lab_pb = np.array([row['p_baseline'] for row in coarse_rows])
lab_qb = np.array([row['q_shifted'] for row in coarse_rows])
{k: {'fine': f_divergence(lab_p, lab_q, k), 'coarse': f_divergence(lab_pb, lab_qb, k)} for k in kinds}


## Pitfalls

Coarse graining is not the same as projection onto a parametric submanifold. It is a map between observation alphabets. A sufficient statistic can be many-to-one because the discarded variation is conditionally parameter-free.

Euclidean distance on probabilities is a tempting baseline, but it is not invariant under this criterion. It depends on the chosen observational resolution.

The Fisher metric agreement among standard f-divergences is local. The global divergences can disagree strongly near the boundary of the simplex, especially when one distribution assigns zero or tiny mass to an outcome.


## Final Sanity Checks

The final check records four invariants: monotonicity under the chapter's partition, equality in the sufficient-statistic case, strict loss after changing within-block conditionals, and Fisher-length preservation under a sufficient embedding.


In [ ]:
# Numerical sanity checks for the chapter thread.
r = [normalize([0.35, 0.65]), normalize([0.2, 0.5, 0.3]), normalize([0.6, 0.25, 0.15])]
p_suff = np.concatenate([pb[a] * r[a] for a in range(len(blocks))])
qb_target = normalize([0.24, 0.31, 0.45])
q_suff = np.concatenate([qb_target[a] * r[a] for a in range(len(blocks))])
q_break = q_suff.copy()
q_break[2:5] = qb_target[1] * normalize([0.55, 0.25, 0.20])

q_small = normalize([0.28, 0.44, 0.28])
v_small = np.array([0.08, -0.05, -0.03])
R = [normalize([0.7, 0.3]), normalize([0.2, 0.5, 0.3]), normalize([0.4, 0.6])]
p_embed = np.concatenate([q_small[i] * R[i] for i in range(3)])
v_embed = np.concatenate([v_small[i] * R[i] for i in range(3)])
fisher_small = float(np.sum(v_small ** 2 / q_small))
fisher_embed = float(np.sum(v_embed ** 2 / p_embed))

sanity = {
    'chapter': '03-invariant-geometry-of-probability-distributions',
    'source_orientation': {'printed_pages': '51-70', 'pdf_pages': '63-81'},
    'monotonicity_gaps': {labels[i]: float(full[i] - coarse_vals[i]) for i in range(len(labels))},
    'all_monotonicity_gaps_nonnegative': bool(np.all(full + 1e-12 >= coarse_vals)),
    'sufficient_statistic_kl_residual': float(abs(f_divergence(p_suff, q_suff, 'kl') - f_divergence(pb, qb_target, 'kl'))),
    'broken_conditional_kl_gap': float(f_divergence(p_suff, q_break, 'kl') - f_divergence(pb, qb_target, 'kl')),
    'fisher_embedding_residual': float(abs(fisher_small - fisher_embed)),
    'artifact_files': {},
}
for name in ['coarse_graining_partition.png', 'f_divergence_monotonicity.png', 'sufficient_statistic_equality.png', 'fisher_embedding_invariance.png', 'alpha_divergence_simplex.html', 'applied_lab_sensor_bins.json']:
    path = ARTIFACT_DIR / name
    sanity['artifact_files'][name] = {'exists': path.exists(), 'bytes': path.stat().st_size if path.exists() else 0}
sanity['checks_passed'] = bool(
    sanity['all_monotonicity_gaps_nonnegative']
    and sanity['sufficient_statistic_kl_residual'] < 1e-10
    and sanity['broken_conditional_kl_gap'] > 1e-6
    and sanity['fisher_embedding_residual'] < 1e-12
    and all(v['bytes'] > 100 for v in sanity['artifact_files'].values())
)
(ARTIFACT_DIR / 'final_sanity.json').write_text(json.dumps(sanity, indent=2), encoding='utf-8')
sanity


## Takeaways

1. Coarse graining is the basic test for whether a divergence respects loss of observational information.
2. Decomposable invariant divergences are exactly f-divergences in the finite simplex setting, aside from low-dimensional exceptional behavior.
3. Equality under coarse graining is the signature of sufficiency: the discarded conditional distribution carries no parameter information.
4. Every standard f-divergence has the Fisher information as its second-order metric.
5. Chentsov's theorem turns this into a uniqueness principle: up to scale, Fisher information is the invariant Riemannian metric.
6. On positive measures, square-root coordinates reveal the invariant metric as Euclidean, while normalized probabilities sit on a sphere.


## Standalone Synthesis

The chapter focus is: Invariance, coarse graining, sufficient statistics, f-divergences, KL properties, Fisher information, and positive measures.

Key computational translations:

- coarse graining cannot increase distinguishability for invariant divergences. In the notebook this is represented by a concrete array, curve, surface, or metric object, then checked numerically so the reader can see the invariant rather than infer it from prose alone.
- f-divergences share a common convex-generator form. In the notebook this is represented by a concrete array, curve, surface, or metric object, then checked numerically so the reader can see the invariant rather than infer it from prose alone.
- Fisher information is the local metric selected by invariance. In the notebook this is represented by a concrete array, curve, surface, or metric object, then checked numerically so the reader can see the invariant rather than infer it from prose alone.
- positive measures widen the geometry beyond normalized distributions. In the notebook this is represented by a concrete array, curve, surface, or metric object, then checked numerically so the reader can see the invariant rather than infer it from prose alone.

How to read the visual sequence:

- coarse-graining map from a fine simplex to a smaller simplex. This visual should be read as evidence for a geometric claim, not as decoration; the surrounding code records the quantities that make the claim testable.
- gallery of KL, chi-square, Hellinger, and alpha divergences. This visual should be read as evidence for a geometric claim, not as decoration; the surrounding code records the quantities that make the claim testable.
- local Fisher metric ellipses on the simplex. This visual should be read as evidence for a geometric claim, not as decoration; the surrounding code records the quantities that make the claim testable.
- data-processing inequality experiment with repeated random channels. This visual should be read as evidence for a geometric claim, not as decoration; the surrounding code records the quantities that make the claim testable.

This standalone synthesis is intentionally explicit about the contract between explanation, computation, and artifact. The reader should be able to close the PDF and still reconstruct the chapter's working picture: what the objects are, which coordinates are being used, what the visual is meant to reveal, and which numerical check guards the interpretation. When a divergence, metric, projection, or learning path appears, the notebook treats it as an inspectable mechanism. Changing the toy parameters is part of the lesson because it separates robust geometric behavior from an accidental drawing. If the saved check remains stable, the picture is carrying the idea; if the check fails, the diagram has become misleading and the model assumptions need to be revisited.

This standalone synthesis is intentionally explicit about the contract between explanation, computation, and artifact. The reader should be able to close the PDF and still reconstruct the chapter's working picture: what the objects are, which coordinates are being used, what the visual is meant to reveal, and which numerical check guards the interpretation. When a divergence, metric, projection, or learning path appears, the notebook treats it as an inspectable mechanism. Changing the toy parameters is part of the lesson because it separates robust geometric behavior from an accidental drawing. If the saved check remains stable, the picture is carrying the idea; if the check fails, the diagram has become misleading and the model assumptions need to be revisited.


## Course Standard Note

**Source span:** printed pages 51-70; PDF pages 68-87. The PDF is used only for source orientation, not as a required companion while reading this standalone notebook. The final_sanity evidence for this chapter is stored under `artifacts/chapter-03` using the chapter's local sanity JSON naming convention.


In [ ]:
from pathlib import Path

def _discover_book_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'artifacts').exists():
            return candidate
    raise RuntimeError('Could not locate book root')

BOOK_ROOT = _discover_book_root(Path.cwd())
BOOK_ROOT
